# PyMOL scenes for three case-study proteins

30133_1, 16608_1 and 17512_1: B-factor PDBs and `.pml` scenes colored by |error|. Residues: GT present, prediction mask True, no proline. Output folder: `pml/` (override with `CASE_PML_OUT`).

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

ROOT = Path("../../../data/analysis")
RBMRB = Path("../../../data/rBMRB")
GT_PATH = RBMRB / "cluster_30p/PANAV/test.json"
ENS_PATH = ROOT / "Fig4/results/predictions_N_seed_ensemble_10seed.json"
FEAT_PATH = ROOT / "Fig4/all_features/per_residue_records_full.json"
STRUCT_DIR = RBMRB / "cluster_30p/structures_split_filtered/test"

PML_DIR = Path(os.environ.get("CASE_PML_OUT", "pml"))   # set CASE_PML_OUT to write a dry run elsewhere
PML_DIR.mkdir(parents=True, exist_ok=True)
BIG = 3.0     # ppm, "large error"
VMAX = 5.0    # ppm, top of the colour scale (same as write_case_pml.ipynb)

gt = json.load(open(GT_PATH))
key_of = {v: k for k, v in gt["entry_ID"].items()}
ens = json.load(open(ENS_PATH))
feat = json.load(open(FEAT_PATH))

## Cases

In [ ]:
def span(sequence, first):
    # consecutive stretch: span("QQRLIF", 40) -> ["Q40", "Q41", "R42", "L43", "I44", "F45"]
    return [f"{a}{first + i}" for i, a in enumerate(sequence)]


CASES = [
    dict(entry="30133_1", name="2''-aminoglycoside nucleotidyltransferase", item="apo_open_closed",
         story="Apo ANT(2'')-Ia (aminoglycoside resistance): the enzyme exchanges between a closed (70%) and an "
               "open (30%) conformation; the aminoglycoside-recognition segment W125 and I129-D137 (sticks) is "
               "semi-flexible, and substrate binding repositions the NT and C-terminal domains. Errors are high "
               "across the whole protein.",
         citation="Bacot-Davis VR et al. Protein Sci. 26, 1852-1863 (2017)",
         key=["W125"] + span("IWDYFYYAD", 129)),
    dict(entry="16608_1", name="Insulin", item="molten_CA_junction",
         story="Proinsulin: the CA junction (PC2 cleavage at K64-R65) and the A1-A8 helix are molten (attenuated "
               "13C secondary shifts); the A4, A7, A8 and A11 amides are lost to conformational broadening. These "
               "segments carry receptor site 1 (A1-A3, A5) and the A6-A11 disulfide.",
         citation="Yang Y et al. J. Biol. Chem. 285, 7847-7851 (2010)",
         key=span("ALEGSLQKRGIVEQCCT", 57) + ["C76"]),           # C27-C35 + A1-A8 (A1 = G66) + A11
    dict(entry="17512_1", name="DNA primase", item="helix_caps",
         story="S. aureus DnaG primase C-terminal domain (all-helical): half of its large errors sit at helix N-cap, "
               "N1, C1 or C-cap positions (sticks), which make up 23% of the residues. Over the whole test set the "
               "large-error rate is 11% at N-caps vs 3.8% inside helices (adjusted OR 2.3).",
         citation="Aurora R, Rose GD. Protein Sci. 7, 21-38 (1998) (helix capping)",
         key="helix_caps"),
]

## Error per residue, key residues, summary

In [ ]:
def gated_error(entry):
    # {resnum: |ensemble mean - GT|} over gated residues (GT not None, ensemble mask True, no proline)
    k = key_of[entry]
    seq, N = gt["sequence"][k], gt["N"][k]
    n = len(seq)
    mean = np.asarray(ens["mean"][entry][:n], float)
    mask = np.asarray(ens["mask"][entry][:n], bool)
    return {i + 1: abs(mean[i] - float(N[i])) for i in range(n)
            if N[i] is not None and mask[i] and seq[i] != "P"}


def helix_caps(entry):
    seq = gt["sequence"][key_of[entry]]
    ss = {r["resnum"]: r.get("ss_class") for r in feat[entry]}
    caps = set()
    for i in range(1, len(seq) + 1):
        if ss.get(i) == "helix" and ss.get(i - 1) != "helix":
            caps |= {i - 1, i}                     # N-cap, N1
        if ss.get(i) == "helix" and ss.get(i + 1) != "helix":
            caps |= {i, i + 1}                     # C1, C-cap
    return [f"{seq[r - 1]}{r}" for r in sorted(caps) if 1 <= r <= len(seq)]


def key_resnums(case):
    # residue-type check against the construct sequence; returns the residue numbers
    seq = gt["sequence"][key_of[case["entry"]]]
    labels = helix_caps(case["entry"]) if case["key"] == "helix_caps" else case["key"]
    nums = [int(lab[1:]) for lab in labels]
    wrong = [lab for lab, n in zip(labels, nums) if seq[n - 1] != lab[0]]
    assert not wrong, f"{case['entry']}: residue type mismatch {wrong}"
    return nums


for c in CASES:
    seq = gt["sequence"][key_of[c["entry"]]]
    err = gated_error(c["entry"])
    nums = key_resnums(c)
    vals = np.array(list(err.values()))
    n_big = int((vals >= BIG).sum())
    n_big_key = sum(1 for r in nums if err.get(r, 0) >= BIG)
    n_key_assigned = sum(1 for r in err if r in set(nums))
    c["err"], c["resnums"] = err, nums
    c["summary"] = (f"median |error| {np.median(vals):.2f} ppm; large errors (>= {BIG} ppm) {n_big}/{len(vals)}, "
                    f"of which {n_big_key} at the {len(nums)} key residues "
                    f"(key residues = {n_key_assigned / len(vals):.0%} of assigned residues)")
    print(f"{c['entry']} ({c['name']}): {c['summary']}")
    print("    large errors at key residues:",
          ", ".join(f"{seq[r - 1]}{r}({err[r]:.1f})" for r in nums if err.get(r, 0) >= BIG) or "none")

## Write the B-factor PDBs and the PyMOL scenes

In [ ]:
PML = '''# {title}
# story: {story}
# source: {citation}
# data: {summary}
# colour: |pred - exp| 15N error, blue (0 ppm) -> white -> red (>= {vmax} ppm); grey = no data (proline / unassigned)
# sticks: key residues from the source ({n_key}); sphere = backbone amide N (the measured nucleus)
bg_color white
set ray_opaque_background, 0
set orthoscopic, on

set specular, 0
set reflect, 0
set ambient, 1
set depth_cue, 0
set ray_trace_mode, 1
set ray_trace_depth_factor, 1
set ray_trace_disco_factor, 1
set ray_trace_gain, 0.005
set ray_shadow, off
set antialias, 2
set cartoon_oval_length, 1
set cartoon_rect_length, 1

load {pdb}, prot
hide everything, prot
show cartoon, prot
select prot_nodata, prot and b < -0.5
select prot_hasdata, prot and b > -0.5
spectrum b, white_red, prot_hasdata, minimum=0.0, maximum={vmax}
color grey70, prot_nodata

select key_sites, prot and resi {resi}
show sticks, key_sites and not name C+N+O
show spheres, key_sites and name CA
hide everything, hydro
set stick_radius, 0.35, key_sites
set sphere_scale, 0.35, key_sites and name CA
# label key_sites and name CA, "%s%s" % (resn, resi)

deselect
orient prot
# orient key_sites   # zoom on the key residues
#ray 1800, 1500
#png {png}, dpi=300
'''


def write_bfactor_pdb(entry, err, path):
    # |error| of each gated residue in the B-factor column; -1 for residues without a gated error
    lines = []
    for line in open(STRUCT_DIR / f"{entry}.pdb"):
        if line.startswith(("ATOM", "HETATM")):
            line = line[:60] + f"{err.get(int(line[22:26]), -1.0):6.2f}" + line[66:]
        lines.append(line)
    path.write_text("".join(lines))


for c in CASES:
    pdb = f"{c['entry']}_acerr_bfactor.pdb"
    write_bfactor_pdb(c["entry"], c["err"], PML_DIR / pdb)
    out = PML_DIR / f"render_{c['entry']}_{c['item']}.pml"
    out.write_text(PML.format(title=f"{c['entry']} {c['name']}", story=c["story"], citation=c["citation"],
                              summary=c["summary"], vmax=VMAX, n_key=len(c["resnums"]), pdb=pdb,
                              resi="+".join(map(str, c["resnums"])), png=out.stem + ".png"))
    print("wrote", out.name, "and", pdb)